# Visual Test for Color Conversions

This notebook provides a simple visual test for color conversions. 

## Converting from Other Color Systems to sRGB

* Choose a color system on the right. Not all color systems are available in CSS.
* The first component value of the color system is set by the slider. 
* The second value increases down the row (top to bottom). 
* The third value increases down the row (left to right).
* The background color of the table cells are that of the chosen color system. The text in each cell is colored in the converted sRGB value. If both this implementation, the browser in which you're viewing this notebook, and your display are compliant, the text should be invisible or almost invisible.
* If sRGB CSS gamut mapping is enabled, the sRGB color is gamut-mapped to the sRGB gamut using the **CSS gamut map** algorithm. Browsers may implement gamut mapping differently. 
* The foreground and background colors can be swapped for easier viewing.

## Converting from sRGB to Other Color Systems
* Pick an sRGB color from the color picker and it will be converted into the color system specified above. No gamut mapping will be performed.
* The text in the large square should be barely visible.

In [1]:
from musculus.metadata.color import RGBAColor
from musculus.util.colorsystem import *
from textwrap import dedent
from math import nan  # pyright: ignore[reportGeneralTypeIssues]
from traitlets.traitlets import link, dlink
from ipywidgets.widgets import (
    HTML,
    Checkbox,
    FloatSlider,
    ColorPicker,
    Box,
    VBox,
    Label,
    BoundedFloatText,
    Layout,
    Dropdown,
)
from ipywidgets.widgets.widget_float import SliderStyle

global_style = """<style>
    table {
        table-layout: fixed;
        max-width: 800px
    }
    th, td {
        min-width: 30px;
        max-width: 30px;
    }
    tr, th, td {
        font-size: 10px;
        line-height: 16px;
        font-family: monospace;
        font-weight: bold;
        text-align: center;
        min-height: 30px;
        max-height: 30px;
    }
    </style>
    """


row_count = col_count = 11
v2_min = 0.0
v2_max = 1.0
v3_min = 0.0
v3_max = 1.0
use_gamut_map = False
swap = False

COLOR_SYNTAX = "color({system} {v1:f} {v2:f} {v3:f} / {alpha:f})"
FUNCTION_SYNTAX = "{system}({v1:f} {v2:f} {v3:f} / {alpha:f})"


def format_color_css(system, v1, v2, v3, alpha=1.0):
    match system:
        case ("srgb"
            | "display-p3"
            | "a98-rgb"
            | "prophoto-rgb"
            | "rec2020"
            | "xyz-d50"
            | "xyz-d65"
            | "srgb-linear"
            | "display-p3-linear"
            | "rec2020"
            | "xyz-d50"
            | "xyz-d65"):
            color_syntax = COLOR_SYNTAX
        case _:
            color_syntax = FUNCTION_SYNTAX

    return color_syntax.format(
                **{
                    "system": system,
                    "v1": v1,
                    "v2": v2,
                    "v3": v3,
                    "alpha": alpha
                }
            )

def make_table():
    after = "after" if use_gamut_map else "before"
    system: ColorSystem = model_chooser.value
    v1 = slider.value
    rows = []
    headings = []
    v3s = []
    for col_id in range(col_count):
        v3 = (col_id / (col_count - 1)) * (v3_max - v3_min) + v3_min
        headings.append(f'<th scope="col">{v3:.2f}</th>')
        v3s.append(v3)
    heading_trs = "".join(headings)
    for row_id in range(row_count):
        v2 = (row_id / (row_count - 1)) * (v2_max - v2_min) + v2_min
        cells = []
        for v3 in v3s:
            if system == "wavelength":
                r, g, b = convert(system, ColorSystem.SRGB, (v1,))
                css_srgb_value = f"rgb({r:.2%} {g:.2%} {b:.2%})"
            elif use_gamut_map:
                r, g, b = convert_into_gamut(
                    system, ColorSystem.SRGB, (v1, v2, v3), min_l=nan, max_l=nan
                )
                css_srgb_value = f"rgb({r:.2%},{g:.2%},{b:.2%})"
            else:
                r, g, b = convert(system, ColorSystem.SRGB, (v1, v2, v3))
                css_srgb_value = f"rgb({r:.2%} {g:.2%} {b:.2%})"
            
            css_model_value = format_color_css(system, v1, v2, v3)
            text = f"r:{r:.0%} g:{g:.0%} b:{b:.0%}"
            roundtripped = convert(ColorSystem.SRGB, system, (r, g, b))
            roundtripped_text = ", ".join(f"{float(f):02f}" for f in roundtripped)
            c, m, y, k = convert(system, ColorSystem.CMYK, (v1, v2, v3))
            cmyk_text = f"C:{c:.0%} M:{m:.0%} Y:{y:.0%} K:{k:.0%}"
            title = dedent(
                f"""
            Model value: {(v1, v2, v3)}
            Roundtripped: ({roundtripped_text})
            CSS value: {css_model_value}
            sRGB value ({after} gamut mapping): {text}
            CMYK: {cmyk_text}
            """
            )
            if swap:
                text_color = css_model_value
                background_color = css_srgb_value
            else:
                text_color = css_srgb_value
                background_color = css_model_value
            cells.append(
                f"<td style='color: {text_color}; background: {background_color}' title='{title}'>"
                f"<span style='text-decoration: underline {RGBAColor.from_model(system, roundtripped)!s}'>{text}</span></td>"
            )
        tds = "".join(cells)
        rows.append(f'<tr><th scope="col">{v2:.2f}</th>{tds}</tr>')
    trs = "".join(rows)

    t = f"<table><thead><tr><th>{system}</th>{heading_trs}</tr></thead><tbody>{trs}</tbody></table>"
    return t


h = HTML(layout=Layout(width="1fr"))


def update_table(change={}):
    global v2_max, v2_min, v3_max, v3_min, color_syntax, use_gamut_map, swap
    use_gamut_map = gamut_map_checkbox.value
    _v1_max = _v2_max = _v3_max = 1.0
    _v1_min = _v2_min = _v3_min = 0.0
    swap = swap_checkbox.value
    match model_chooser.value:
        case "hsl" | "hwb":
            _v1_max = 360
            _v2_max = _v3_max = 100
        case "lab":
            _v1_max = 100
            _v2_min = _v3_min = -125
            _v2_max = _v3_max = 125
        case "lch":
            _v1_max = 100
            _v2_max = 100
            _v3_max = 360
        case "oklab":
            _v2_min = _v3_min = -0.4
            _v2_max = _v3_max = 0.4
        case "oklch":
            _v2_min = 0
            _v2_max = 1
            _v3_min = 0
            _v3_max = 360
        case "wavelength":
            _v1_min = 380
            _v1_max = 780
        case _:
            pass
    if change.get("owner", None) == model_chooser:
        slider_min.value = _v1_min
        slider_max.value = _v1_max
        slider_min.value = _v1_min
        v2_max = float(_v2_max)
        v2_min = float(_v2_min)
        v3_max = float(_v3_max)
        v3_min = float(_v3_min)
    h.value = global_style + make_table()


model_chooser = Dropdown(
    value="srgb",
    options={k: v for k, v in ColorSystem.__members__.items() if v != "cmyk"},
    description="Color System",
    layout=Layout(max_width="200px"),
)
gamut_map_checkbox = Checkbox(
    indent=False, value=False, description="Use sRGB CSS gamut mapping"
)
swap_checkbox = Checkbox(
    indent=False, value=False, description="Swap foreground / background"
)
slider = FloatSlider(
    orientation="horizontal",
    readout=True,
    step=0.01,
    style=SliderStyle(description_width="0px"),
    layout=Layout(width="100%"),
)
slider_min = BoundedFloatText(
    description="", min=-1e10, value=0.0, layout=Layout(max_width="60px")
)
slider_max = BoundedFloatText(
    description="", max=1e10, value=1.0, layout=Layout(max_width="60px")
)
slider_min.value = 0
dlink((slider_min, "value"), (slider_max, "min"))
dlink((slider_max, "value"), (slider_min, "max"))
link((slider_min, "value"), (slider, "min"))
link((slider_max, "value"), (slider, "max"))
slider.observe(update_table, "value")
model_chooser.observe(update_table, "value")
gamut_map_checkbox.observe(update_table, "value")
swap_checkbox.observe(update_table, "value")
sliders = Box([slider_min, slider, slider_max])
app = Box(
    [
        h,
        VBox(
            [model_chooser, gamut_map_checkbox, swap_checkbox],
            layout=Layout(width="fit-content"),
        ),
    ],
    layout=Layout(width="100%", grid_template_columns="60px 1fr 60px"),
)


def color_observe(*_):
    system = model_chooser.value
    picker_value = color_picker.value
    rgba_color = RGBAColor.parse(picker_value)
    *rgb, _ = rgba_color.to_floats()
    converted = convert(ColorSystem.SRGB, system, rgb)
    converted_text = ", ".join(f"{v:.2f}" for v in converted)
    try:
        v1, v2, v3 = converted
    except ValueError:
        (v1,) = converted
        v2 = v3 = 0.0
    css_model_value = format_color_css(system, v1, v2, v3)
    if swap:
        text_color = picker_value
        background_color = css_model_value
    else:
        text_color = css_model_value
        background_color = picker_value
    h2.value = (
        f"""<div style="display: flex; justify-content: center; align-items: center; width: 200px; height: 200px;"""
        f"""background: {background_color}; color: {text_color}; font-family: monospace">"""
        f"""<span>{system} value:<br>{converted_text}</span></div>"""
    )


h2 = HTML()
color_picker = ColorPicker(description="sRGB Color")

color_picker.observe(color_observe, "value")
model_chooser.observe(color_observe, "value")
swap_checkbox.observe(color_observe, "value")

display(Label("From Color Systems to sRGB"), app, sliders)
display(Label("From sRGB to Color Systems"), color_picker, h2)
color_observe()
update_table()

Label(value='From Color Systems to sRGB')

Box(children=(HTML(value='', layout=Layout(width='1fr')), VBox(children=(Dropdown(description='Color System', …

Box(children=(BoundedFloatText(value=0.0, layout=Layout(max_width='60px'), max=1.0, min=-10000000000.0), Float…

Label(value='From sRGB to Color Systems')

ColorPicker(value='black', description='sRGB Color')

HTML(value='')

## Color Interpolation

Pick the two color stops and compare the color strip with the gradinent below. Interpolation (especially hue interpolation) may produce intermediate colors outside the sRGB gamut which are not guaranteed to be displayed or converted correctly, either by code or by browser (yes, browsers can be quite inconsistent in terms of color and gradients).

In [2]:
from musculus.metadata.color import *
from musculus.util.colorsystem import *
from ipywidgets.widgets import (
    HBox,
    VBox,
    BoundedIntText,
    ColorPicker,
    Dropdown,
    HTML,
)
from typing import cast
from musculus.util.linalg import Tuple3


from_color_picker = ColorPicker(description="From Color", value="#ff0000")
from_alpha_input = BoundedIntText(
    description="alpha",
    value=255,
    min=0,
    max=255,
    step=1,
)
to_color_picker = ColorPicker(description="To Color", value="#0000ff")
to_alpha_input = BoundedIntText(
    description="alpha",
    value=255,
    min=0,
    max=255,
    step=1,
)
interpolation_dropdown = Dropdown(
    value="oklab",
    options=list(INTERPOLATION_COLOR_SYSTEMS),
    description="Interpolation",
)
hue_method = Dropdown(
    value="shorter",
    options=list(HueInterpolationMethod),
    description="Hue Method",
)
h3 = HTML()
display(HBox([interpolation_dropdown, hue_method]))
display(
    HBox(
        [
            VBox([from_color_picker, from_alpha_input]),
            VBox([to_color_picker, to_alpha_input]),
        ]
    )
)
display(h3)

NUM_INTERPOLATIONS = 20

def update_interpolation(*_):
    interpolation = interpolation_dropdown.value
    hue_interpolation_method = hue_method.value
    from_alpha = from_alpha_input.value
    from_rgba = RGBAColor(RGBAColor(from_color_picker.value), alpha=from_alpha)
    from_color = cast(Tuple3, convert(ColorSystem.SRGB, interpolation, from_rgba.to_floats()[0:3]))

    to_alpha = to_alpha_input.value
    to_rgba = RGBAColor(RGBAColor(to_color_picker.value), alpha=to_alpha)
    to_color = cast(Tuple3, convert(ColorSystem.SRGB, interpolation, to_rgba.to_floats()[0:3]))
    t = ""
    for i in range(0, NUM_INTERPOLATIONS + 1):
        interpolated, interpolated_alpha = interpolate(
            interpolation,
            i / NUM_INTERPOLATIONS,
            interpolation,
            from_color,
            from_alpha / 255,
            interpolation,
            to_color,
            to_alpha / 255,
            hue_interpolation_method=hue_interpolation_method,
        )
        color_str=format_color_css(interpolation, *interpolated, alpha=float(interpolated_alpha))
        t += (
            f"""<div style="display: flex; justify-content: center; align-items: center; flex: auto; height: 100px;"""
            f"""background: {color_str};"""
            f""" font-family: monospace" title="{color_str}">"""
            f"""<span style="background: #ffffff80">{i / NUM_INTERPOLATIONS:.0%}</span></div>"""
        )
    # from_css = format_color_css(interpolation, *from_color)
    # to_css = format_color_css(interpolation, *to_color)
    from_css = from_rgba.to_hex_rrggbbaa()
    to_css = to_rgba.to_hex_rrggbbaa()
    if interpolation in POLAR_INTERPOLATION_SYSTEMS:
        hue_method.disabled = False
        gradient_string = f"to right in {interpolation} {hue_interpolation_method} hue, {from_css!s}, {to_css!s}"
    else:
        hue_method.disabled = True
        gradient_string = f"to right in {interpolation}, {from_css!s}, {to_css!s}"
    h3.value = (
        f"""<div style="display: flex; width: 100%; height:200px; """
        f"""background-image: linear-gradient({gradient_string});">{t}</div>"""
    )


from_color_picker.observe(update_interpolation, "value")
from_alpha_input.observe(update_interpolation, "value")
to_color_picker.observe(update_interpolation, "value")
to_alpha_input.observe(update_interpolation, "value")
interpolation_dropdown.observe(update_interpolation, "value")
hue_method.observe(update_interpolation, "value")
update_interpolation()

HTML(value='')